# CSELReader Demo

Demo for [`CSELReader`](https://github.com/diyclassics/latincy-readers), a reader for TEI-XML files from the [Corpus Scriptorum Ecclesiasticorum Latinorum](https://github.com/OpenGreekAndLatin/csel-dev) digital edition published by the Open Greek and Latin Project.

## Setup

In [1]:
from latincyreaders import CSELReader, AnnotationLevel

In [ ]:
CSEL_PATH = "/path/to/csel-dev/data"  # local clone of https://github.com/OpenGreekAndLatin/csel-dev
reader = CSELReader(root=CSEL_PATH)

## File Discovery

In [3]:
reader.fileids()[:8]

['stoa0040.stoa001.opp-lat1.xml']

In [4]:
len(reader.fileids())

1

## Metadata

In [5]:
list(reader.headers())[:5]

[{'filename': 'stoa0040.stoa001.opp-lat1.xml',
  'title': 'Confessiones',
  'author': 'Augustine',
  'cts_urn': 'urn:cts:latinLit:stoa0040.stoa001.opp-lat1'}]

## Core Interface

### texts()

In [6]:
next(reader.texts())[:500]

'Magnus es, domine, et laudabilis valde: magna virtus tua et sapientiae tuae non est numerus.\n\nEt quomodo invocabo deum meum, deum et dominum meum, quoniam utique in me ipsum eum invocabo.\n\nRecordari volo praeteritas foeditates meas et carnales corruptiones animae meae, non quod eas amem, sed ut amem te, deus meus.\n\nQuid mihi erat in isto furto et quid te, domine deus meus, imitari volui scelerate?'

### docs()

In [7]:
doc = next(reader.docs())
doc

Magnus es , domine , et laudabilis ualde : magna uirtus tua et sapientiae tuae non est numerus . Et quomodo inuocabo deum meum , deum et dominum meum , quoniam utique in me ipsum eum inuocabo . Recordari uolo praeteritas foeditates meas et carnales corruptiones animae meae , non quod eas amem , sed ut amem te , deus meus . Quid mihi erat in isto furto et quid te , domine deus meus , imitari uolui scelerate ?

In [8]:
doc._.metadata

{'filename': 'stoa0040.stoa001.opp-lat1.xml',
 'path': '/Volumes/fiona/work/code/diy/latincy-v3/latincy-readers/tests/fixtures/csel/stoa0040.stoa001.opp-lat1.xml',
 'title': 'Confessiones',
 'author': 'Augustine',
 'cts_urn': 'urn:cts:latinLit:stoa0040.stoa001.opp-lat1'}

### sents()

In [9]:
sent = next(reader.sents())
sent

Magnus es , domine , et laudabilis ualde :

In [10]:
from itertools import islice

sent_counts = {}
sent_refs = []
for sent in doc.sents:
    chapter = next(
        (ch for ch in doc.spans["chapters"] if ch.start <= sent.start < ch.end),
        None,
    )
    if chapter:
        idx = chapter._.citation
        sent_counts[idx] = sent_counts.get(idx, 0) + 1
        sent_refs.append((f"{idx}.{sent_counts[idx]}", sent.text))

for ref, text in islice(sent_refs, 8):
    print(f"{ref}: {text[:60]}")

book 1, section 1.1: Magnus es , domine , et laudabilis ualde :
book 1, section 1.2: magna uirtus tua et sapientiae tuae non est numerus .
book 1, section 2.1: Et quomodo inuocabo deum meum , deum et dominum meum , quoni
book 2, section 1.1: Recordari uolo praeteritas foeditates meas et carnales corru
book 2, section 2.1: Quid mihi erat in isto furto et quid te , domine deus meus ,


### tokens()

In [11]:
list(reader.tokens())[:8]

[Magnus, es, ,, domine, ,, et, laudabilis, ualde]

In [12]:
tok = next(reader.tokens())
tok.text, tok.lemma_, tok.pos_

('Magnus', 'magnus', 'ADJ')

## CSEL Features

### chapters()

In [13]:
for citation, text in list(reader.chapters(as_text=True))[:8]:
    print(f"{citation}: {text[:80]}")

book 1, section 1: Magnus es, domine, et laudabilis valde: magna virtus tua et sapientiae tuae non 
book 1, section 2: Et quomodo invocabo deum meum, deum et dominum meum, quoniam utique in me ipsum 
book 2, section 1: Recordari volo praeteritas foeditates meas et carnales corruptiones animae meae,
book 2, section 2: Quid mihi erat in isto furto et quid te, domine deus meus, imitari volui scelera


### Chapter spans

In [14]:
for chapter in list(doc.spans["chapters"])[:8]:
    print(f"{chapter._.citation}: {chapter.text[:80]}")

book 1, section 1: Magnus es , domine , et laudabilis ualde : magna uirtus tua et sapientiae tuae n
book 1, section 2: . Et quomodo inuocabo deum meum , deum et dominum meum , quoniam utique in me ip
book 2, section 1: inuocabo . Recordari uolo praeteritas foeditates meas et carnales corruptiones a
book 2, section 2: meus . Quid mihi erat in isto furto et quid te , domine deus meus , imitari uolu


### CTS URN metadata

In [15]:
for meta in list(reader.headers())[:8]:
    print(meta["cts_urn"])
    print(f"  {meta['author']}: {meta['title']}")

urn:cts:latinLit:stoa0040.stoa001.opp-lat1
  Augustine: Confessiones


### Annotation levels

In [16]:
# TOKENIZE: fast, no model required
reader_tok = CSELReader(root=CSEL_PATH, annotation_level=AnnotationLevel.TOKENIZE)
tok_doc = next(reader_tok.docs())
[(t.text, t.lemma_) for t in tok_doc[:8]]

[('Magnus', ''),
 ('es', ''),
 (',', ''),
 ('domine', ''),
 (',', ''),
 ('et', ''),
 ('laudabilis', ''),
 ('valde', '')]